<a href="https://colab.research.google.com/github/liujh22/learngit/blob/master/Group_B_working_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="在 Colab 中打开"/></a>

# Group B：跨学习阶段脑区 Decoder

本 notebook 检验 `leaf1` 与 `circle1` 的脑区平均空间活动模式能否在学习前后双向迁移。主分析使用 TX105 的学习前与学习后两个 session，不要求跨 session 配准单个神经元。

运行顺序为：下载数据 → 加载 SVD 与行为 → 转换为“成分 × 试次 × 位置” → 构造脑区 40 维空间特征 → 会话内交叉验证与双向迁移 → 统计与绘图。请在 Colab 中从头运行；旧的缓存输出不应视为当前结果。

In [ ]:
# @title Import required packages
from pathlib import Path
import requests
import numpy as np
import matplotlib.pyplot as plt

## 1. 下载本分析需要的数据

这里只下载 TX105 学习前/后 decoder 实际使用的六个文件：两个 SVD 文件、两个脑区标签文件和两个行为文件。已经存在的文件不会重复下载。

In [ ]:
# @title Download the six files used by the TX105 decoder
DATA_ROOT = Path("/content/Zhong_et_al_2025")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

before_session = "TX105_2022_10_08_2"
after_session = "TX105_2022_10_19_2"

REQUIRED_FILES = {
    "TX105_2022_10_08_2_SVD_dec.npy",
    "TX105_2022_10_08_trans.npz",
    "TX105_2022_10_19_2_SVD_dec.npy",
    "TX105_2022_10_19_trans.npz",
    "Beh_unsup_train1_before_learning.npy",
    "Beh_unsup_train1_after_learning.npy",
}

BASE_URL = "https://api.figshare.com/v2"
ARTICLE_ID = 28811129

# Resolve files by name so unrelated supervised-session files are never fetched.
metadata_response = requests.get(
    f"{BASE_URL}/articles/{ARTICLE_ID}",
    timeout=60,
)
metadata_response.raise_for_status()
file_metadata = metadata_response.json()
metadata_by_name = {
    item["name"]: item for item in file_metadata["files"]
}

missing_from_article = REQUIRED_FILES.difference(metadata_by_name)
if missing_from_article:
    raise FileNotFoundError(
        "Figshare article is missing required files: "
        f"{sorted(missing_from_article)}"
    )

for filename in sorted(REQUIRED_FILES):
    destination = DATA_ROOT / filename
    if destination.exists():
        print("Using cached file:", destination)
        continue

    file_id = metadata_by_name[filename]["id"]
    with requests.get(
        f"{BASE_URL}/file/download/{file_id}",
        stream=True,
        timeout=120,
    ) as download_response:
        download_response.raise_for_status()
        with destination.open("wb") as output_file:
            for chunk in download_response.iter_content(1024 * 1024):
                if chunk:
                    output_file.write(chunk)
    print("Downloaded:", destination)

## 2. 加载 SVD 与最小行为字段

`U` 的轴为“成分 × 神经元”，保存每个 session 自己的空间 loading；`V` 的轴为“成分 × 帧”，保存成分随时间的活动。后续只需要行为中的试次数、累计位置和 `WallName` 标签，因此不再创建舔舐、跑速或示例绘图变量。

In [ ]:
# @title Load SVD arrays and the behavioral fields used downstream
def load_svd_session(session_id):
    """Return session-specific spatial loadings U and component activity V."""
    payload = np.load(
        DATA_ROOT / f"{session_id}_SVD_dec.npy",
        allow_pickle=True,
    ).item()
    U = np.asarray(payload["U"])
    V = np.asarray(payload["V"])
    if U.ndim != 2 or V.ndim != 2 or U.shape[0] != V.shape[0]:
        raise ValueError(f"Invalid SVD shapes for {session_id}: {U.shape}, {V.shape}")
    return U, V


U_bef, V_bef = load_svd_session(before_session)
U_aft, V_aft = load_svd_session(after_session)

behavior_before = np.load(
    DATA_ROOT / "Beh_unsup_train1_before_learning.npy",
    allow_pickle=True,
).item()
behavior_after = np.load(
    DATA_ROOT / "Beh_unsup_train1_after_learning.npy",
    allow_pickle=True,
).item()
beh_bef = behavior_before[before_session]
beh_aft = behavior_after[after_session]

ntrials_bef = int(beh_bef["ntrials"])
ntrials_aft = int(beh_aft["ntrials"])
cum_pos_fr_bef = np.asarray(beh_bef["ft_PosCum"])
cum_pos_fr_aft = np.asarray(beh_aft["ft_PosCum"])

print(f"Before: U={U_bef.shape}, V={V_bef.shape}, trials={ntrials_bef}")
print(f"After:  U={U_aft.shape}, V={V_aft.shape}, trials={ntrials_aft}")

## 3. 从连续帧转换到试次和位置

两个 session 共用同一个插值函数。输出保留完整 60 个位置分箱，因为 decoder 需要先使用灰色区域 40–59 做无标签归一化，之后才截取纹理区域 0–39。

In [ ]:
# TRIAL_POSITION_UTILS: shared frame-to-trial-position conversion
import numpy as np
from scipy import interpolate


def activity_by_trial_and_position(
    activity_by_frame,
    cumulative_position,
    n_trials,
    corridor_length=60,
):
    """Interpolate signals from recording frames to trial-by-position bins.

    Parameters
    ----------
    activity_by_frame : array-like
        Two-dimensional signals × frames array. Here the signals are SVD
        components, but the function also accepts reconstructed neurons.
    cumulative_position : array-like
        Monotonically non-decreasing cumulative corridor position per frame.
    n_trials : int
        Number of complete corridor trials represented in the output.
    corridor_length : int, default=60
        Number of integer position bins per trial.

    Returns
    -------
    numpy.ndarray
        Float32 array with shape signals × trials × positions.
    """
    activity_by_frame = np.asarray(activity_by_frame)
    cumulative_position = np.asarray(cumulative_position)
    if activity_by_frame.ndim != 2:
        raise ValueError("activity_by_frame must be a 2-D signals × frames array")

    n_trials = int(n_trials)
    corridor_length = int(corridor_length)
    if n_trials < 1 or corridor_length < 1:
        raise ValueError("n_trials and corridor_length must be positive")

    n_frames = activity_by_frame.shape[1]
    if len(cumulative_position) < n_frames:
        raise ValueError(
            "The behavioral position array has fewer frames than the "
            "neural activity array."
        )

    source_positions = cumulative_position[:n_frames]

    # Pauses create repeated cumulative positions. interp1d requires unique
    # coordinates, so retain the first neural sample at every position.
    unique_positions, unique_indices = np.unique(
        source_positions,
        return_index=True,
    )
    if len(unique_positions) < 2:
        raise ValueError("At least two unique positions are required")

    target_positions = np.arange(n_trials * corridor_length)
    interpolated = np.empty(
        (activity_by_frame.shape[0], len(target_positions)),
        dtype=np.float32,
    )
    for signal_index, signal in enumerate(activity_by_frame):
        model = interpolate.interp1d(
            unique_positions,
            signal[unique_indices],
            bounds_error=False,
            fill_value="extrapolate",
        )
        interpolated[signal_index] = model(target_positions)

    return interpolated.reshape(
        activity_by_frame.shape[0],
        n_trials,
        corridor_length,
    )

In [ ]:
# @title Convert both sessions to components × trials × 60 positions
component_activity_trial_pos_bef = activity_by_trial_and_position(
    V_bef,
    cum_pos_fr_bef,
    n_trials=ntrials_bef,
    corridor_length=60,
)
component_activity_trial_pos_aft = activity_by_trial_and_position(
    V_aft,
    cum_pos_fr_aft,
    n_trials=ntrials_aft,
    corridor_length=60,
)

# Keep all 60 position bins until gray-region normalization in the decoder.
assert component_activity_trial_pos_bef.shape == (
    U_bef.shape[0],
    ntrials_bef,
    60,
)
assert component_activity_trial_pos_aft.shape == (
    U_aft.shape[0],
    ntrials_aft,
    60,
)
print("Before component tensor:", component_activity_trial_pos_bef.shape)
print("After component tensor:", component_activity_trial_pos_aft.shape)

## 4. 构造脑区特征并定义 decoder

每个脑区先对该 session 的 `U` loading 按神经元求平均，再与成分活动相乘，得到“试次 × 60 位置”的脑区平均曲线。这等价于重建该脑区全部神经元后再求平均，但避免生成巨大的三维神经元数组。

每个 session 和脑区独立使用灰色位置 40–59 的总体均值与标准差做无标签归一化，再保留纹理位置 0–39 作为 40 维输入。模型为 `StandardScaler` 加类别平衡逻辑回归。

In [ ]:
# CROSS_SESSION_AREA_DECODER_UTILS: cross-session area-decoder utilities
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

AREA_CODES = {
    "V1": (8,),
    "medial": (0, 1, 2, 9),
    "lateral": (5, 6),
    "anterior": (3, 4),
}
ANALYSIS_AREAS = ("V1", "medial", "lateral", "anterior", "visual_all")


def make_area_masks(iarea):
    """Create masks for four visual areas and their union from the raw area codes."""
    iarea = np.asarray(iarea).ravel()
    masks = {name: np.isin(iarea, codes) for name, codes in AREA_CODES.items()}
    masks["visual_all"] = np.logical_or.reduce(list(masks.values()))
    return masks


def area_position_features(U, component_activity, iarea, area_name, eps=1e-12):
    """Reconstruct mean area activity and normalize it using that session's gray corridor."""
    U = np.asarray(U)
    component_activity = np.asarray(component_activity)
    iarea = np.asarray(iarea).ravel()

    if U.ndim != 2 or component_activity.ndim != 3:
        raise ValueError("U and component_activity must be 2-D and 3-D")
    if U.shape[0] != component_activity.shape[0]:
        raise ValueError("component axis mismatch between U and component_activity")
    if U.shape[1] != len(iarea):
        raise ValueError("neuron axis mismatch between U and iarea")
    if component_activity.shape[2] < 60:
        raise ValueError("component_activity must contain 60 position bins")

    masks = make_area_masks(iarea)
    if area_name not in masks:
        raise ValueError(f"unknown area: {area_name}")

    area_mask = masks[area_name]
    if not area_mask.any():
        raise ValueError(f"area {area_name} contains no neurons")

    # Average the session-specific U loadings only within the requested area.
    # Multiplying this mean loading by component activity is algebraically
    # equivalent to reconstructing every area neuron and then averaging them,
    # but it avoids a very large neurons × trials × positions array.
    area_loading = U[:, area_mask].mean(axis=1)
    area_activity = np.einsum("c,ctp->tp", area_loading, component_activity)

    # Estimate one label-free baseline per session and area from gray positions.
    # This removes session-wide offset and scale differences before transfer.
    gray = area_activity[:, 40:60]
    gray_mean = float(gray.mean())
    gray_std = float(gray.std())
    if not np.isfinite(gray_std) or gray_std < eps:
        raise ValueError(f"gray_std is invalid for area {area_name}: {gray_std}")

    normalized = (area_activity - gray_mean) / gray_std
    X = normalized[:, :40]
    if not np.isfinite(X).all():
        raise ValueError(f"non-finite features for area {area_name}")

    return X, {
        "area": area_name,
        "n_neurons": int(area_mask.sum()),
        "gray_mean": gray_mean,
        "gray_std": gray_std,
    }


def encode_leaf_circle_labels(behavior):
    """Keep only circle1 and leaf1, encoded as circle1=0 and leaf1=1."""
    wall_name = np.asarray(behavior["WallName"]).astype(str)
    keep = np.isin(wall_name, ["circle1", "leaf1"])
    labels = (wall_name[keep] == "leaf1").astype(int)
    if set(np.unique(labels)) != {0, 1}:
        raise ValueError("both circle1 and leaf1 must be present")
    return labels, keep


def make_decoder():
    """Create a standardized logistic-regression decoder with fixed hyperparameters."""
    return make_pipeline(
        StandardScaler(),
        LogisticRegression(
            C=1.0,
            class_weight="balanced",
            max_iter=5000,
            random_state=0,
        ),
    )


def _cv_balanced_accuracy(X, y):
    """Estimate within-session balanced accuracy with stratified cross-validation."""
    # Within-session CV refits the scaler and classifier on every training fold.
    # Consequently, no held-out fold contributes to StandardScaler parameters.
    counts = np.bincount(y, minlength=2)
    n_splits = int(min(5, counts.min()))
    if n_splits < 2:
        raise ValueError("each class needs at least two trials")
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=0)
    return float(
        cross_val_score(
            make_decoder(),
            X,
            y,
            cv=cv,
            scoring="balanced_accuracy",
        ).mean()
    )


def evaluate_transfer(
    X_before,
    y_before,
    X_after,
    y_after,
    n_bootstrap=2000,
    n_permutations=1000,
):
    """Fit only on the first source session and evaluate transfer on the second target session."""
    X_before = np.asarray(X_before, dtype=float)
    X_after = np.asarray(X_after, dtype=float)
    y_before = np.asarray(y_before, dtype=int)
    y_after = np.asarray(y_after, dtype=int)

    if (
        X_before.ndim != 2
        or X_after.ndim != 2
        or X_before.shape[1] != X_after.shape[1]
    ):
        raise ValueError("source and target features must be 2-D with equal columns")
    if X_before.shape[1] != 40:
        raise ValueError("final feature count must be 40")
    if len(X_before) != len(y_before) or len(X_after) != len(y_after):
        raise ValueError("feature and label trial counts must match")
    if not np.isfinite(X_before).all() or not np.isfinite(X_after).all():
        raise ValueError("feature matrices must be finite")

    # Fit preprocessing and classification only on the source session.
    # The target session is used only for scoring; neither step is refit there.
    pipeline = make_decoder().fit(X_before, y_before)
    predicted = pipeline.predict(X_after)
    probability = pipeline.predict_proba(X_after)[:, 1]
    observed = balanced_accuracy_score(y_after, predicted)

    # Resample target trials within each class to preserve both labels in every
    # bootstrap replicate and quantify uncertainty in transfer accuracy.
    bootstrap_rng = np.random.default_rng(2)
    class_indices = [np.flatnonzero(y_after == value) for value in (0, 1)]
    if any(len(indices) == 0 for indices in class_indices):
        raise ValueError("target session must contain both classes")
    bootstrap_scores = np.empty(n_bootstrap)
    for index in range(n_bootstrap):
        sampled = np.concatenate(
            [
                bootstrap_rng.choice(indices, size=len(indices), replace=True)
                for indices in class_indices
            ]
        )
        bootstrap_scores[index] = balanced_accuracy_score(
            y_after[sampled],
            predicted[sampled],
        )

    # Shuffle only source labels, refit the full pipeline, and keep the target
    # session untouched to form the cross-session chance distribution.
    permutation_rng = np.random.default_rng(1)
    permutation_scores = np.empty(n_permutations)
    for index in range(n_permutations):
        shuffled = permutation_rng.permutation(y_before)
        null_model = make_decoder().fit(X_before, shuffled)
        permutation_scores[index] = balanced_accuracy_score(
            y_after,
            null_model.predict(X_after),
        )

    metrics = {
        "before_cv_balanced_accuracy": _cv_balanced_accuracy(X_before, y_before),
        "after_cv_balanced_accuracy": _cv_balanced_accuracy(X_after, y_after),
        "transfer_balanced_accuracy": float(observed),
        "transfer_roc_auc": float(roc_auc_score(y_after, probability)),
        "bootstrap_ci_low": float(np.quantile(bootstrap_scores, 0.025)),
        "bootstrap_ci_high": float(np.quantile(bootstrap_scores, 0.975)),
        "permutation_p": float(
            (1 + np.sum(permutation_scores >= observed))
            / (1 + n_permutations)
        ),
    }
    artifacts = {
        "pipeline": pipeline,
        "predicted": predicted,
        "probability": probability,
        "confusion_matrix": confusion_matrix(
            y_after,
            predicted,
            labels=[0, 1],
        ),
        "bootstrap_scores": bootstrap_scores,
        "permutation_scores": permutation_scores,
    }
    return metrics, artifacts


def evaluate_bidirectional_transfer(
    X_before,
    y_before,
    X_after,
    y_after,
    n_bootstrap=2000,
    n_permutations=1000,
):
    """Fit independent models in both directions and return direction-labeled results."""
    # Each call creates and fits a new source-specific scaler/classifier.
    forward_metrics, forward_artifacts = evaluate_transfer(
        X_before,
        y_before,
        X_after,
        y_after,
        n_bootstrap=n_bootstrap,
        n_permutations=n_permutations,
    )
    reverse_metrics, reverse_artifacts = evaluate_transfer(
        X_after,
        y_after,
        X_before,
        y_before,
        n_bootstrap=n_bootstrap,
        n_permutations=n_permutations,
    )
    return (
        {
            "before_to_after": forward_metrics,
            "after_to_before": reverse_metrics,
        },
        {
            "before_to_after": forward_artifacts,
            "after_to_before": reverse_artifacts,
        },
    )

## 5. 运行会话内与双向跨会话解码

`Before CV` 和 `After CV` 分别只在自己的 session 内做分层交叉验证，每一折都会重新拟合 scaler 和 classifier。迁移分析则在全部源 session 试次上拟合一次，并直接测试目标 session；目标标签只用于最终评分。

两个迁移方向使用完全独立的模型。每个脑区报告 balanced accuracy、ROC AUC、目标试次 bootstrap 95% 置信区间、源标签 permutation p 值和混淆矩阵。

In [ ]:
# @title Bidirectional area-population decoding across learning

# Area-label filenames omit the final recording-block number.
date_key_bef = "_".join(before_session.split("_")[:-1])
date_key_aft = "_".join(after_session.split("_")[:-1])

iarea_bef = np.load(
    DATA_ROOT / f"{date_key_bef}_trans.npz",
    allow_pickle=True,
)["iarea"]
iarea_aft = np.load(
    DATA_ROOT / f"{date_key_aft}_trans.npz",
    allow_pickle=True,
)["iarea"]

# Use identical binary labels in both sessions: circle1=0 and leaf1=1.
y_bef, keep_bef = encode_leaf_circle_labels(beh_bef)
y_aft, keep_aft = encode_leaf_circle_labels(beh_aft)

# Stop early if SVD, anatomical, behavioral, and trial axes do not agree.
assert len(iarea_bef) == U_bef.shape[1]
assert len(iarea_aft) == U_aft.shape[1]
assert len(keep_bef) == component_activity_trial_pos_bef.shape[1]
assert len(keep_aft) == component_activity_trial_pos_aft.shape[1]

rows = []
cross_session_artifacts = {}

# Build a separate shared 40-position feature space for every anatomical area.
for area in ANALYSIS_AREAS:
    X_bef_all, meta_bef = area_position_features(
        U_bef,
        component_activity_trial_pos_bef,
        iarea_bef,
        area,
    )
    X_aft_all, meta_aft = area_position_features(
        U_aft,
        component_activity_trial_pos_aft,
        iarea_aft,
        area,
    )

    X_bef = X_bef_all[keep_bef]
    X_aft = X_aft_all[keep_aft]

    # The two directions fit independent source-session pipelines.
    direction_metrics, direction_artifacts = evaluate_bidirectional_transfer(
        X_bef,
        y_bef,
        X_aft,
        y_aft,
        n_bootstrap=2000,
        n_permutations=1000,
    )
    metrics = direction_metrics["before_to_after"]
    reverse_metrics = direction_metrics["after_to_before"]

    rows.append(
        {
            "area": area,
            "n_neurons_before": meta_bef["n_neurons"],
            "n_neurons_after": meta_aft["n_neurons"],
            "n_trials_before": len(y_bef),
            "n_trials_after": len(y_aft),
            "gray_mean_before": meta_bef["gray_mean"],
            "gray_std_before": meta_bef["gray_std"],
            "gray_mean_after": meta_aft["gray_mean"],
            "gray_std_after": meta_aft["gray_std"],
            **metrics,
            "after_to_before_balanced_accuracy": reverse_metrics[
                "transfer_balanced_accuracy"
            ],
            "after_to_before_roc_auc": reverse_metrics["transfer_roc_auc"],
            "after_to_before_bootstrap_ci_low": reverse_metrics[
                "bootstrap_ci_low"
            ],
            "after_to_before_bootstrap_ci_high": reverse_metrics[
                "bootstrap_ci_high"
            ],
            "after_to_before_permutation_p": reverse_metrics["permutation_p"],
        }
    )
    cross_session_artifacts[area] = {
        **direction_artifacts,
        "X_before": X_bef,
        "X_after": X_aft,
        "y_before": y_bef,
        "y_after": y_aft,
    }
    print(
        f"{area:>10s} | "
        f"before CV={metrics['before_cv_balanced_accuracy']:.3f} | "
        f"after CV={metrics['after_cv_balanced_accuracy']:.3f} | "
        f"before→after={metrics['transfer_balanced_accuracy']:.3f} "
        f"(p={metrics['permutation_p']:.4f}) | "
        f"after→before={reverse_metrics['transfer_balanced_accuracy']:.3f} "
        f"(p={reverse_metrics['permutation_p']:.4f})"
    )

# Store scalar metrics in one table; retain predictions and null distributions
# separately because they are needed by the diagnostic figures below.
cross_session_results = pd.DataFrame(rows)
display(
    cross_session_results[
        [
            "area",
            "n_neurons_before",
            "n_neurons_after",
            "n_trials_before",
            "n_trials_after",
            "before_cv_balanced_accuracy",
            "after_cv_balanced_accuracy",
            "transfer_balanced_accuracy",
            "transfer_roc_auc",
            "bootstrap_ci_low",
            "bootstrap_ci_high",
            "permutation_p",
            "after_to_before_balanced_accuracy",
            "after_to_before_roc_auc",
            "after_to_before_bootstrap_ci_low",
            "after_to_before_bootstrap_ci_high",
            "after_to_before_permutation_p",
        ]
    ].round(3)
)

## 6. 可视化结果

第一张图单独比较学习前和学习后的会话内 CV；第二张图比较两个迁移方向及 bootstrap 区间；第三张图把四种准确率放到同一坐标中。最后两组图展示方向特异的混淆矩阵和 permutation 零分布。

会话内 CV 高但 transfer 低，表示两个阶段各自包含信息，但平均空间表征格式可能改变。需要同时查看置信区间和 permutation 检验，不能只根据点估计下结论。

In [ ]:
# @title Visualize bidirectional decoding across learning

area_labels = cross_session_results["area"].tolist()
x = np.arange(len(area_labels))

# Compare information available within each learning stage. These bars are
# cross-validated within one session and are not cross-session transfer scores.
within_session_specs = [
    ("before_cv_balanced_accuracy", "Before CV"),
    ("after_cv_balanced_accuracy", "After CV"),
]
fig, ax = plt.subplots(figsize=(8, 4.5))
within_width = 0.34
within_offsets = (-within_width / 2, within_width / 2)
for offset_value, (column, label) in zip(
    within_offsets, within_session_specs
):
    bars = ax.bar(
        x + offset_value,
        cross_session_results[column],
        width=within_width,
        label=label,
    )
    ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=8)
ax.axhline(0.5, color="black", linestyle="--", linewidth=1, label="chance")
ax.set_xticks(x, area_labels, rotation=20)
ax.set_ylim(0, 1)
ax.set_ylabel("Cross-validated balanced accuracy")
ax.set_title("Within-session decoding by learning stage")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

# Plot both transfer directions with target-trial bootstrap confidence intervals.
forward_transfer = cross_session_results[
    "transfer_balanced_accuracy"
].to_numpy()
forward_low = cross_session_results["bootstrap_ci_low"].to_numpy()
forward_high = cross_session_results["bootstrap_ci_high"].to_numpy()
reverse_transfer = cross_session_results[
    "after_to_before_balanced_accuracy"
].to_numpy()
reverse_low = cross_session_results[
    "after_to_before_bootstrap_ci_low"
].to_numpy()
reverse_high = cross_session_results[
    "after_to_before_bootstrap_ci_high"
].to_numpy()

fig, ax = plt.subplots(figsize=(8, 4.5))
offset = 0.08
ax.errorbar(
    x - offset,
    forward_transfer,
    yerr=np.vstack(
        [
            forward_transfer - forward_low,
            forward_high - forward_transfer,
        ]
    ),
    fmt="o",
    capsize=4,
    linewidth=1.5,
    label="Before → after",
)
ax.errorbar(
    x + offset,
    reverse_transfer,
    yerr=np.vstack(
        [
            reverse_transfer - reverse_low,
            reverse_high - reverse_transfer,
        ]
    ),
    fmt="o",
    capsize=4,
    linewidth=1.5,
    label="After → before",
)
ax.axhline(0.5, color="black", linestyle="--", linewidth=1, label="chance")
ax.set_xticks(x, area_labels, rotation=20)
ax.set_ylim(0, 1)
ax.set_ylabel("Balanced accuracy")
ax.set_title("Bidirectional cross-session decoding")
ax.legend()
plt.tight_layout()
plt.show()

# Place within-session baselines and transfer scores on the same scale so that
# a low transfer score can be distinguished from low decodability in a session.
fig, ax = plt.subplots(figsize=(9, 4.5))
width = 0.2
metric_specs = [
    ("before_cv_balanced_accuracy", "Before CV"),
    ("after_cv_balanced_accuracy", "After CV"),
    ("transfer_balanced_accuracy", "Before → after"),
    ("after_to_before_balanced_accuracy", "After → before"),
]
offsets = (-1.5 * width, -0.5 * width, 0.5 * width, 1.5 * width)
for offset_value, (column, label) in zip(offsets, metric_specs):
    ax.bar(
        x + offset_value,
        cross_session_results[column],
        width=width,
        label=label,
    )
ax.axhline(0.5, color="black", linestyle="--", linewidth=1)
ax.set_xticks(x, area_labels, rotation=20)
ax.set_ylim(0, 1)
ax.set_ylabel("Balanced accuracy")
ax.set_title("Within-session and bidirectional cross-session decoding")
ax.legend()
plt.tight_layout()
plt.show()

# Confusion matrices reveal whether transfer errors favor circle1 or leaf1.
direction_specs = [
    ("before_to_after", "Before → after"),
    ("after_to_before", "After → before"),
]

fig, axes = plt.subplots(
    len(direction_specs),
    len(area_labels),
    figsize=(3.2 * len(area_labels), 6),
)
for row_index, (direction_key, direction_label) in enumerate(direction_specs):
    for column_index, area in enumerate(area_labels):
        ax = axes[row_index, column_index]
        matrix = cross_session_artifacts[area][direction_key][
            "confusion_matrix"
        ]
        image = ax.imshow(matrix, cmap="Blues")
        for true_class in range(2):
            for predicted_class in range(2):
                ax.text(
                    predicted_class,
                    true_class,
                    matrix[true_class, predicted_class],
                    ha="center",
                    va="center",
                    color="black",
                )
        ax.set_xticks([0, 1], ["circle1", "leaf1"], rotation=30)
        ax.set_yticks([0, 1], ["circle1", "leaf1"])
        ax.set_xlabel("Predicted")
        ax.set_ylabel("True")
        ax.set_title(f"{area}\n{direction_label}")
fig.colorbar(image, ax=axes, shrink=0.75)
plt.show()

# Compare observed transfer against source-label permutation null scores.
fig, axes = plt.subplots(
    len(direction_specs),
    len(area_labels),
    figsize=(3.2 * len(area_labels), 6),
)
for row_index, (direction_key, direction_label) in enumerate(direction_specs):
    for column_index, area in enumerate(area_labels):
        ax = axes[row_index, column_index]
        direction_artifacts = cross_session_artifacts[area][direction_key]
        null_scores = direction_artifacts["permutation_scores"]
        if direction_key == "before_to_after":
            observed_column = "transfer_balanced_accuracy"
            p_column = "permutation_p"
        else:
            observed_column = "after_to_before_balanced_accuracy"
            p_column = "after_to_before_permutation_p"
        observed = cross_session_results.loc[
            cross_session_results["area"] == area,
            observed_column,
        ].iloc[0]
        p_value = cross_session_results.loc[
            cross_session_results["area"] == area,
            p_column,
        ].iloc[0]
        ax.hist(null_scores, bins=25, color="0.75", edgecolor="white")
        ax.axvline(observed, color="tab:red", linewidth=2)
        ax.set_title(f"{area}\n{direction_label}: p={p_value:.4f}")
        ax.set_xlabel("Permuted balanced accuracy")
        ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

## 解读限制

这里的 40 个特征是一条脑区平均空间活动曲线，而不是跨 session 配准的单神经元 population vector。显著迁移支持脑区平均空间模式具有跨阶段稳定性；迁移较低并不能证明脑区没有刺激信息，因为不同神经元或亚群的信号可能在平均时抵消。

当前 notebook 只分析一只小鼠的两个 session，不能直接推广为群体水平的学习效应。完整结果应在 Colab 中从头运行并保存后再解释。